# Gemini Enterprise Agent Platform Pipelines での AutoML を使用した継続的トレーニング

**学習目標:**
1. Agent Platform AutoML の事前構築済みコンポーネントの使用方法を学ぶ
1. BigQuery をデータソースとして使用し、これらのコンポーネントで Agent Platform AutoML パイプラインを構築する方法を学ぶ
1. Agent Platform AutoML パイプラインをコンパイル、アップロード、実行する方法を学ぶ


このラボでは、モデルのトレーニング、チューニング、デプロイを行う **Agent Platform AutoML AI** サービスをオーケストレーションする Agent Platform AutoML パイプラインを構築、デプロイ、実行します。

## セットアップ

In [ ]:
import os

from google.cloud import aiplatform

%load_ext google.cloud.bigquery

In [ ]:
REGION = "us-central1"
PROJECT = !(gcloud config get-value project)
PROJECT = PROJECT[0]

os.environ["PROJECT"] = PROJECT

### BigQuery データ

KFP ウォークスルーラボを実施していない場合は、以下のセルを実行して、このラボに必要なデータを含む BigQuery データセットとテーブルを作成する必要があります。

**注:** すでに `<PROJECT_ID>.covertype_dataset.covertype` の BigQuery テーブルに Covertype データがある場合は、**パイプライン設計の理解** に進んで構いません。

In [ ]:
%%bash

DATASET_LOCATION=US
DATASET_ID=covertype_dataset
TABLE_ID=covertype
DATA_SOURCE=gs://asl-public-data/data/covertype/dataset.csv
SCHEMA=Elevation:INTEGER,\
Aspect:INTEGER,\
Slope:INTEGER,\
Horizontal_Distance_To_Hydrology:INTEGER,\
Vertical_Distance_To_Hydrology:INTEGER,\
Horizontal_Distance_To_Roadways:INTEGER,\
Hillshade_9am:INTEGER,\
Hillshade_Noon:INTEGER,\
Hillshade_3pm:INTEGER,\
Horizontal_Distance_To_Fire_Points:INTEGER,\
Wilderness_Area:STRING,\
Soil_Type:STRING,\
Cover_Type:INTEGER

bq --location=$DATASET_LOCATION --project_id=$PROJECT mk --dataset $DATASET_ID

bq --project_id=$PROJECT --dataset_id=$DATASET_ID load \
--source_format=CSV \
--skip_leading_rows=1 \
--replace \
$TABLE_ID \
$DATA_SOURCE \
$SCHEMA

## パイプライン設計の理解

パイプラインによって実装されるワークフローは、Python ベースのドメイン固有言語（DSL）を使用して定義されます。パイプラインの DSL は、以下で生成する `pipeline/pipeline_automl.py` ファイル内に記述されます。

パイプラインの DSL は、ファイルパスや接続文字列などの環境固有の設定をハードコードしないように設計されています。これらの設定は、一連の環境変数を介してパイプラインコードに提供されます。

## パイプラインの構築とデプロイ

### 演習

以下のパイプラインを完成させてください:

In [ ]:
%%writefile ./pipeline/pipeline_automl.py
# Copyright 2021 Google LLC

# Licensed under the Apache License, Version 2.0 (the "License"); you may not
# use this file except in compliance with the License. You may obtain a copy of
# the License at

# https://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS"
# BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either
# express or implied. See the License for the specific language governing
# permissions and limitations under the License.

"""Kubeflow Covertype パイプライン。"""

import os

from google_cloud_pipeline_components.v1.automl.training_job import (
    AutoMLTabularTrainingJobRunOp,
)
from google_cloud_pipeline_components.v1.dataset import TabularDatasetCreateOp
from google_cloud_pipeline_components.v1.endpoint import (
    EndpointCreateOp,
    ModelDeployOp,
)

from kfp import dsl

PIPELINE_ROOT = os.getenv("PIPELINE_ROOT")
PROJECT = os.getenv("PROJECT")
DATASET_SOURCE = os.getenv("DATASET_SOURCE")
PIPELINE_NAME = os.getenv("PIPELINE_NAME", "covertype")
DISPLAY_NAME = os.getenv("MODEL_DISPLAY_NAME", PIPELINE_NAME)
TARGET_COLUMN = os.getenv("TARGET_COLUMN", "Cover_Type")
SERVING_MACHINE_TYPE = os.getenv("SERVING_MACHINE_TYPE", "e2-standard-8")
# Agent Platform AI AutoML コンポーネントのイメージバージョン
KFP_IMAGE = "gcr.io/ml-pipeline/google-cloud-pipeline-components:2.16.1"

@dsl.pipeline(
    name=f"{PIPELINE_NAME}-automl-pipeline",
    description=f"AutoML Pipeline for {PIPELINE_NAME}",
    pipeline_root=PIPELINE_ROOT,
)
def create_pipeline():

    dataset_create_task = TabularDatasetCreateOp(
        # TODO
    )

    automl_training_task = AutoMLTabularTrainingJobRunOp(
        # TODO
    )

    # コンテナイメージのバージョンをオーバーライド
    automl_training_task.set_container_image(KFP_IMAGE)

    endpoint_create_task = EndpointCreateOp(
        # TODO
    )

    model_deploy_task = ModelDeployOp(  # pylint: disable=unused-variable
        # TODO
    )


### パイプラインのコンパイル

まず、パイプラインコンパイラに渡される環境変数を定義することから始めましょう。

In [ ]:
ARTIFACT_STORE = f"gs://{PROJECT}-kfp-artifact-store"
PIPELINE_ROOT = f"{ARTIFACT_STORE}/pipeline"
DATASET_SOURCE = f"bq://{PROJECT}.covertype_dataset.covertype"

%env PIPELINE_ROOT={PIPELINE_ROOT}
%env PROJECT={PROJECT}
%env REGION={REGION}
%env DATASET_SOURCE={DATASET_SOURCE}

`ARTIFACT_STORE` が作成されていることを確認し、作成されていない場合は作成しましょう。

In [ ]:
!gcloud storage ls | grep ^{ARTIFACT_STORE}/$ || gcloud storage buckets create --location={REGION} {ARTIFACT_STORE}

#### CLI コンパイラを使用したパイプラインのコンパイル

次のコマンドを使用して、生成した Python ファイルからパイプラインを YAML 記述にコンパイルします。

In [ ]:
PIPELINE_YAML = "covertype_automl_pipeline.yaml"

### 演習

`kfp dsl compile` コマンドラインを使用してパイプラインをコンパイルします。

In [ ]:
# TODO

**注:** Python SDK を使用してパイプラインをコンパイルすることもできます。

```python
from kfp import compiler

compiler.Compiler().compile(
    pipeline_func=create_pipeline, 
    package_path=PIPELINE_YAML,
)

```

結果としてパイプラインファイルが生成されます。

In [ ]:
!head {PIPELINE_YAML}

### パイプラインパッケージのデプロイ

### 演習

`aiplatform.PipelineJob` を使用してパイプラインを Agent Platform にアップロードして実行します。

In [ ]:
# TODO

Copyright 2021 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.